In [11]:
import argparse
import csv
import gzip
import json
import logging
import random
import time
from dataclasses import asdict, dataclass, field
from datetime import datetime
from io import BytesIO
from pathlib import Path
from typing import Optional
from urllib.parse import urlparse
 
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
 
# ── constants ──────────────────────────────────────────────────────────────────
PAGE_TIMEOUT      = 20


In [12]:
def build_driver(headless: bool = False) -> webdriver.Chrome:
    opts = Options()

    # ── anti-bot ───────────────────────────────────────────────────────────────
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)

    # ── VPS / headless safe flags ──────────────────────────────────────────────
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")                  # required on VPS
    opts.add_argument("--disable-dev-shm-usage")       # prevents memory crash on VPS
    opts.add_argument("--disable-gpu")                 # no GPU on VPS
    opts.add_argument("--window-size=1280,900")
    opts.add_argument("--disable-extensions")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=opts)

    # ── allow downloads in headless mode (Chrome blocks by default) ────────────
    if headless:
        driver.execute_cdp_cmd(
            "Page.setDownloadBehavior",
            {"behavior": "allow"},
        )

    return driver

In [ ]:
# def human_scroll(driver, element, max_scrolls=20):
#     last_height = 0
    
#     for _ in range(max_scrolls):
#         scroll_amount = random.randint(150, 800)

#         driver.execute_script(
#             "arguments[0].scrollBy(0, arguments[1]);",
#             element,
#             scroll_amount
#         )

#         time.sleep(random.uniform(2, 4))
        
#         new_height = driver.execute_script(
#             "return arguments[0].scrollHeight",
#             element
#         )
#         print(f"New height : {new_height}")
        
#         if new_height == last_height:
#             # One final slow scroll to absolute bottom
#             print(f"  [scroll] Scroll make sure 3 times")
#             for _ in range(3):
#                 driver.execute_script("""
#                     window.scrollTo({
#                         top: document.body.scrollHeight,
#                         behavior: 'smooth'
#                     });
#                 """)
#                 new_height = driver.execute_script("return document.body.scrollHeight")
#                 time.sleep(5)
#             last_height = new_height
#             if new_height == last_height:
#                 print("  [scroll] Reached stable bottom.")
#                 break
#         last_height = new_height

In [25]:
def human_scroll(driver, element):
    import random
    import time

    # 🔸 Use container scroll height (NOT document.body)
    last_height = driver.execute_script(
        "return arguments[0].scrollHeight", element
    )
    
    current_position = 0

    while True:
        # 🔸 Random scroll chunk
        chunk = random.randint(300, 700)
        current_position += chunk
        print(f"current_pos is : {current_position}")

        # 🔸 Scroll INSIDE container
        driver.execute_script("""
            arguments[0].scrollTo({
                top: arguments[1],
                behavior: 'smooth'
            });
        """, element, current_position)

        # 🔸 Small human pause
        time.sleep(random.uniform(0.4, 1.2))

        # 🔸 Occasional reading pause
        if random.random() < 0.2:
            pause = random.uniform(1.5, 3.0)
            print(f"  [scroll] Natural reading pause: {pause:.1f}s")
            time.sleep(pause)

        # 🔸 Occasional scroll back
        if random.random() < 0.1:
            scroll_back = random.randint(50, 150)
            print(f"  [scroll] Natural scroll back: {scroll_back}")
            
            driver.execute_script("""
                arguments[0].scrollTo({
                    top: arguments[1],
                    behavior: 'smooth'
                });
            """, element, current_position - scroll_back)

            time.sleep(random.uniform(0.3, 0.6))

        # 🔸 Check container height growth
        new_height = driver.execute_script(
            "return arguments[0].scrollHeight", element
        )

        # 🔸 If reached bottom of loaded content
        if current_position >= new_height:
            print("  [scroll] Scroll past current bottom")

            time.sleep(3)

            new_height = driver.execute_script(
                "return arguments[0].scrollHeight", element
            )

            if new_height == last_height:
                print("  [scroll] Final bottom stabilization")

                # 🔸 Force bottom scroll a few times
                for _ in range(3):
                    driver.execute_script("""
                        arguments[0].scrollTo({
                            top: arguments[0].scrollHeight,
                            behavior: 'smooth'
                        });
                    """, element)

                    time.sleep(2)

                final_height = driver.execute_script(
                    "return arguments[0].scrollHeight", element
                )

                if final_height == last_height:
                    print("  [scroll] Reached stable bottom.")
                    break

            # 🔸 Update height and continue
            last_height = new_height

In [26]:
driver = build_driver()
driver.get("https://www.google.com/maps/@-8.6605824,115.1762432,3675&hl=en")
time.sleep(30)

wait = WebDriverWait(driver, PAGE_TIMEOUT)
try:
    property_name = "Starloka Saba Bali"
    name_input = wait.until(
        EC.element_to_be_clickable((By.NAME, "q"))
    )
    name_input.clear()
    for i in property_name:
        name_input.send_keys(i)
        time.sleep(random.uniform(0.2, 0.9))
    name_input.send_keys(Keys.ENTER)
except Exception as e:
    print("Error:", e)
time.sleep(30)

# try:
#     # Locate the Reviews tab button by its classes
#     reviews_button = wait.until(
#         EC.element_to_be_clickable(
#             (By.CSS_SELECTOR, "button.hh2c6.G7m0Af[role='tab']")
#         )
#     )
#     # Click the button
#     reviews_button.click()
#     print("Clicked the Reviews tab button!")

# except Exception as e:
#     print("Error:", e)
# time.sleep(10)

try:
    # Locate the Reviews tab button by aria-label (stable selector)
    driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ESCAPE)

# Locate Reviews tab
    reviews_tab = wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//button[@role='tab' and @data-tab-index='2']")
        )
    )

    # Only click if not already active
    if reviews_tab.get_attribute("aria-selected") != "true":
        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            reviews_tab
        )

        try:
            wait.until(EC.element_to_be_clickable(
                (By.XPATH, "//button[@role='tab' and @data-tab-index='2']")
            )).click()
        except:
            driver.execute_script("arguments[0].click();", reviews_tab)
    print("Clicked the Reviews tab button!")
    time.sleep(3)

    driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ESCAPE)

    sort_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[contains(@aria-label, 'Urutkan')]")
        )
    )

    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        sort_btn
    )

    try:
        sort_btn.click()
    except:
        driver.execute_script("arguments[0].click();", sort_btn)

    menu = wait.until(
        EC.presence_of_element_located((By.XPATH, "//div[@role='menu']"))
    )

    # 🔸 Step 3: Click "Terbaru" (Newest)
    newest_option = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//div[@role='menuitemradio']//div[text()='Terbaru']")
        )
    )
    try:
        newest_option.click()
    except:
        driver.execute_script("arguments[0].click();", newest_option)
    print("Clicked the 'Newest' review sort option!")
except Exception as e:
    print("Error:", e)
time.sleep(5)

scroll_box = wait.until(EC.presence_of_element_located((
    By.XPATH, "//div[contains(@class,'m6QErb') and contains(@class,'DxyBCb')]"
)))

human_scroll(driver, scroll_box)
reviews = driver.find_elements(By.XPATH, "//div[contains(@class,'jftiEf')]")

data = []

print(f"Total review is : {len(reviews)}")
ctr = 0
for review in reviews:
    try:
        # Name
        try:
            name = review.get_attribute("aria-label")
        except:
            name = review.find_element(By.CLASS_NAME, "d4r55").text

        # Rating
        try:
            rating = review.find_element(By.CLASS_NAME, "fontBodyLarge").text
        except:
            rating = None

        # Time
        try:
            time_posted = review.find_element(By.CLASS_NAME, "xRkPPb").text
        except:
            time_posted = None

        # Comment
        try:
            comment = review.find_element(By.CLASS_NAME, "wiI7pd").text
        except:
            comment = None

        data.append({
            "name": name,
            "rating": rating,
            "time": time_posted,
            "comment": comment
        })
        ctr += 1
        print(f"Collected comment -> {ctr}")
        time.sleep(2)

    except Exception as e:
        print("Skip one review:", e)

# driver.quit()

Clicked the Reviews tab button!
Clicked the 'Newest' review sort option!
current_pos is : 448
current_pos is : 910
  [scroll] Natural reading pause: 1.8s
current_pos is : 1533
current_pos is : 2193
  [scroll] Natural reading pause: 2.2s
current_pos is : 2550
current_pos is : 3127
current_pos is : 3711
current_pos is : 4354
current_pos is : 4879
current_pos is : 5320
current_pos is : 5770
current_pos is : 6205
current_pos is : 6843
current_pos is : 7489
  [scroll] Natural reading pause: 1.7s
current_pos is : 8145
  [scroll] Natural reading pause: 2.5s
  [scroll] Natural scroll back: 62
current_pos is : 8745
current_pos is : 9419
current_pos is : 9860
current_pos is : 10417
  [scroll] Natural reading pause: 2.3s
  [scroll] Natural scroll back: 147
current_pos is : 10896
current_pos is : 11378
current_pos is : 11989
current_pos is : 12430
current_pos is : 12759
current_pos is : 13144
current_pos is : 13567
current_pos is : 13888
current_pos is : 14446
current_pos is : 14906
current_pos is

In [27]:
import pandas as pd

df = pd.DataFrame(data)

df

,name,rating,time,comment
0,Dede Aryanda,5/5,22 jam lalu di\nGoogle,pengalaman yg tidak dapat dilupakan! dilayani ...
1,Dede Aryanda,5/5,Diedit 22 jam lalu di\nGoogle,bagus sekali dan staff FOnya yg namanya lia sa...
2,Natalya Grebenshchikova,1/5,1 hari lalu di\nGoogle,"Stafnya tidak berbahasa Inggris, harganya tida..."
3,Budi Dharmawan,4/5,1 hari lalu di\nGoogle,None
4,Owner,3/5,5 hari lalu di\nGoogle,Starloka Saba Bali Hotel\nInfo kontak untuk bo...
...,...,...,...,...
150,Sonny Kurniawan,5/5,2 tahun lalu di\nGoogle,"Hotel yg comfort, nyaman, kamar bersih dan ten..."
151,Ygy 2022,5/5,2 tahun lalu di\nGoogle,Nyaman pelayanan baguss 👍 …
152,Shiva Nia Wulandewii,5/5,2 tahun lalu di\nGoogle,Saya menikmati malam-malam terindah dan menena...
153,agus bawa,5/5,2 tahun lalu di\nGoogle,akomodasi yang nyaman dan sesuai budget...


In [28]:
df.to_pickle("google_review.pkl")